# 📝 데이터베이스 기초 과제 LV2 — 표를 잇고, 묶고, 추려 내기

LV1 이 개념 하나씩이었다면, LV2 는 **두 개 이상을 조합**합니다. `JOIN` 과 집계를 함께 쓰고, 서브쿼리로 조건을 만들고, **짝이 없는 행까지 남기는 조인**과 묶은 뒤의 조건(`HAVING`)을 한 문장에 담습니다. 마지막 네 문제(13~16)에서는 조회로 얻은 결과를 **표로 만들어 두고 갱신·삭제·정리**까지 해 봅니다.

전 문항이 **서점 데이터**(고객·도서·주문)로 **sqlite** 에서 풀립니다 — 가입도 키도 필요 없고, 준비 셀만 실행하면 바로 시작할 수 있습니다.

**풀이 방법**은 LV1 과 같습니다. 답안 셀에 코드를 쓰고, 바로 아래 자가채점 셀을 실행해 `✅ 통과!` 를 확인하세요.

> 데이터를 **바꾸는** 문제는 마지막 넷(13~16번)뿐이고, 그것도 여러분이 새로 만드는 `vip_customer` 안에서만 바뀝니다. `customer` · `book` · `orders` 는 끝까지 그대로라, 1~12번의 답은 앞에서 무엇을 실행했든 달라지지 않습니다.

> 13~16번은 **이어지는 한 흐름**입니다 — 만들고(13) · 갱신하고(14) · 추려 내고(15) · 정리합니다(16). 순서대로 푸세요.

## 실습 데이터 — 마당서점

| 표 | 무엇 | 행 수 | 주요 열 |
|---|---|---|---|
| `customer` | 고객 | 10 | `custid`(PK) · `name` · `address` · `phone` |
| `book` | 도서 | 15 | `bookid`(PK) · `bookname` · `publisher` · `price` |
| `orders` | 주문 | 30 | `orderid`(PK) · `custid`(FK) · `bookid`(FK) · `saleprice` · `orderdate` |

`price` 는 **정가**, `saleprice` 는 **실제 판매가**입니다(할인이 있어 다를 수 있습니다). 주문이 한 건도 없는 고객도 있습니다.

In [ ]:
# [제공 코드] — sqlite 실습 준비 (내용은 이해하지 않아도 됩니다 — 실행만 하세요)
import sqlite3
from pathlib import Path

import pandas as pd

ROOT = Path(".") if Path("data").is_dir() else Path("..")
DB_PATH = ROOT / "output" / "bookstore.db"
DB_PATH.parent.mkdir(exist_ok=True)

_conn = None


def get_conn():
    """실습용 sqlite 연결을 하나만 만들어 계속 재사용합니다."""
    global _conn
    if _conn is None:
        _conn = sqlite3.connect(DB_PATH, isolation_level=None)  # 실행하는 즉시 저장(자동 커밋)
        _conn.execute("pragma foreign_keys = on")               # 외래키 검사를 켭니다
    return _conn


def reset_db(script=None):
    """실습 DB를 처음 상태로 되돌립니다. 언제 몇 번을 다시 실행해도 안전합니다."""
    global _conn
    if _conn is not None:
        _conn.close()
        _conn = None
    DB_PATH.unlink(missing_ok=True)
    conn = get_conn()
    if script is not None:
        conn.executescript((ROOT / "data" / script).read_text(encoding="utf-8"))
        conn.execute("pragma foreign_keys = on")
    return conn


def run_sql(sql):
    """결과가 없는 SQL(CREATE·INSERT·UPDATE·DELETE 등)을 실행합니다."""
    get_conn().execute(sql)


def run_query(sql):
    """SELECT 결과를 pandas DataFrame 으로 돌려줍니다."""
    cur = get_conn().execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])


print("sqlite 준비 완료 —", DB_PATH)

In [ ]:
# [제공 코드] — 실습 테이블 전체 리셋 (언제든 다시 실행하면 처음 상태로 돌아갑니다)
reset_db("setup_bookstore.sql")

for t in ["customer", "book", "orders"]:
    n = run_query(f"SELECT count(*) AS n FROM {t}")["n"][0]
    print(f"{t}: {n}행")

## 1. 단골 고객 추리기 (GROUP BY + HAVING)

**배경**: 주문을 3건 이상 한 고객에게 감사 쿠폰을 보냅니다.

**요구사항**: `orders` 를 고객별로 묶어 **주문이 3건 이상인 고객**만 남기고, 주문이 많은 순으로 정렬해 `q1` 에 담으세요. 열은 `custid` 와 주문 건수(별칭 **`order_count`**) 입니다.

**예시**: 결과는 7행입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 묶은 뒤에 거는 조건은 WHERE 가 아니라 HAVING 이다

세부구현:
1. 고객별로 묶고 건수를 센다
2. HAVING 으로 건수 조건을 건다
3. 건수 내림차순으로 정렬한다
```

</details>

In [ ]:
# 묶은 뒤에 거는 조건은 WHERE 가 아니라 HAVING 이다
q1 = run_query("""
SELECT custid,
       count(*) AS order_count
FROM orders
GROUP BY custid
HAVING count(*) >= 3
ORDER BY order_count DESC
""")
display(q1)

<details><summary>해설</summary>

- `WHERE count(*) >= 3` 은 오류입니다 — `WHERE` 는 **묶기 전** 행 하나하나에 걸리는 조건이라 집계 함수를 쓸 수 없습니다.
- `HAVING order_count >= 3` 처럼 별칭으로 써도 sqlite 에서는 동작하지만, 데이터베이스에 따라 안 되는 곳도 있어 `HAVING count(*) >= 3` 이 안전합니다.

</details>

In [ ]:
# [자가채점]
assert len(q1) == 7, f"7행이어야 합니다 (지금 {len(q1)}행)"
assert list(q1.columns) == ["custid", "order_count"], \
    f"열은 custid·order_count 입니다: {list(q1.columns)}"
assert (q1["order_count"] >= 3).all(), "3건 미만인 고객이 섞여 있습니다"
assert q1["order_count"].tolist() == sorted(q1["order_count"].tolist(), reverse=True), \
    "주문이 많은 순으로 정렬하세요"
print("✅ 통과!")

## 2. 고가 출판사 가려내기 (GROUP BY + HAVING — 다른 표)

**배경**: 정가가 대체로 높은 출판사를 골라 별도 협의를 하려 합니다.

**요구사항**: `book` 을 출판사별로 묶어 **평균 정가가 15000 이상**인 출판사만 남기고, 평균 정가가 높은 순으로 정렬해 `q2` 에 담으세요. 열은 `publisher` 와 평균 정가(별칭 **`avg_price`**) 입니다.

**예시**: 결과는 2행이고 첫 행은 **대한미디어** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 1번과 같은 구조지만, 세는 대신 평균을 낸다

세부구현:
1. 출판사별로 묶고 정가의 평균을 구한다
2. HAVING 으로 평균 조건을 건다
3. 평균 내림차순으로 정렬한다
```

</details>

In [ ]:
# HAVING 에는 count 말고도 avg·sum 등 어떤 집계 함수든 올 수 있다
q2 = run_query("""
SELECT publisher,
       avg(price) AS avg_price
FROM book
GROUP BY publisher
HAVING avg(price) >= 15000
ORDER BY avg_price DESC
""")
display(q2)

<details><summary>해설</summary>

- `HAVING` 에는 `count` 말고도 `avg` · `sum` 등 어떤 집계 함수든 올 수 있습니다.
- 출판사가 비어 있는(`NULL`) 책이 있다면 `NULL` 도 하나의 그룹이 됩니다 — 여기서는 모든 책에 출판사가 있습니다.

</details>

In [ ]:
# [자가채점]
assert len(q2) == 2, f"2행이어야 합니다 (지금 {len(q2)}행)"
assert list(q2.columns) == ["publisher", "avg_price"], \
    f"열은 publisher·avg_price 입니다: {list(q2.columns)}"
assert q2["publisher"].tolist() == ['대한미디어', '이상미디어'], \
    f"출판사와 순서를 확인하세요: {q2['publisher'].tolist()}"
assert [round(float(v), 1) for v in q2["avg_price"]] == [25000.0, 19000.0], \
    f"avg_price 는 정가의 평균이어야 합니다: {q2['avg_price'].tolist()}"
print("✅ 통과!")

## 3. 주문에 고객 이름 붙이기 (INNER JOIN)

**배경**: `orders` 에는 고객 번호만 있어 누가 샀는지 알 수 없습니다.

**요구사항**: `orders` 와 `customer` 를 이어 **고객 이름 · 주문번호 · 판매가**를 조회하고 `q3` 에 담으세요. 열 이름은 `name` · `orderid` · `saleprice` 이고, 주문번호 오름차순으로 정렬합니다.

**예시**: 결과는 30행입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 두 표를 외래키 = 기본키 조건으로 잇는다

세부구현:
1. FROM 에 한 표를 적고 JOIN 으로 다른 표를 잇는다
2. ON 에 두 표를 잇는 조건을 적는다
3. 표 별칭을 붙이면 열 이름을 짧게 쓸 수 있다
```

</details>

In [ ]:
# 외래키 = 기본키 조건으로 두 표를 잇는다 — 양쪽에 짝이 있는 행만 남는다
q3 = run_query("""
SELECT c.name, o.orderid, o.saleprice
FROM orders o
JOIN customer c ON c.custid = o.custid
ORDER BY o.orderid
""")
display(q3)

<details><summary>해설</summary>

- `JOIN`(= `INNER JOIN`)은 **양쪽에 짝이 있는 행만** 남깁니다. 모든 주문에는 주인이 있으므로 주문 30건이 그대로 30행이 됩니다.
- 주문이 없는 고객은 이 결과에 나오지 않습니다 — 5번 문제에서 다룹니다.

</details>

In [ ]:
# [자가채점]
assert len(q3) == 30, f"30행이어야 합니다 (지금 {len(q3)}행)"
assert list(q3.columns) == ["name", "orderid", "saleprice"], \
    f"열은 name·orderid·saleprice 입니다: {list(q3.columns)}"
assert q3["orderid"].tolist() == sorted(q3["orderid"].tolist()), "주문번호 오름차순으로 정렬하세요"
print("✅ 통과!")

## 4. 세 표를 이어 판매 내역 만들기 (INNER JOIN — 3표)

**배경**: 주문 내역서에 고객 이름과 **책 제목·정가**까지 함께 싣습니다.

**요구사항**: `orders` · `customer` · `book` 세 표를 이어, `orderid` · `name` · `bookname` · `price` · `saleprice` 와 **정가에서 판매가를 뺀 할인액**(별칭 **`discount`**)을 조회해 `q4` 에 담으세요. 열 순서도 이대로입니다. 할인액이 큰 순, **같으면 주문번호 오름차순**으로 정렬합니다.

**예시**: 결과는 30행 6열입니다. 할인액이 같은 주문이 많으니 두 번째 정렬 기준을 꼭 넣으세요.

<details><summary>힌트</summary>

```text
접근방법:
- JOIN 을 두 번 이어 붙이면 표 세 개가 한 줄로 모인다

세부구현:
1. 주문 표를 가운데 두고 고객·도서를 각각 잇는다
2. 계산 열에 별칭을 붙인다
3. 정렬 기준 두 개를 방향에 맞게 적는다
```

</details>

In [ ]:
# 주문을 가운데 두고 고객·도서를 각각 잇는다. 별칭 discount 는 ORDER BY 에서 쓸 수 있다
q4 = run_query("""
SELECT o.orderid, c.name, b.bookname, b.price, o.saleprice,
       b.price - o.saleprice AS discount
FROM orders o
JOIN customer c ON c.custid = o.custid
JOIN book b ON b.bookid = o.bookid
ORDER BY discount DESC, o.orderid
""")
display(q4)

<details><summary>해설</summary>

- `JOIN` 을 여러 번 이어 쓸 수 있습니다. 가운데에 두 표를 모두 가리키는 표(`orders`)를 두면 자연스럽습니다.
- 별칭 `discount` 는 `ORDER BY` 에서 쓸 수 있습니다(`WHERE` 에서는 쓸 수 없습니다).
- 흔한 실수: `ON` 조건을 빠뜨리면 모든 조합이 만들어져 행이 폭발합니다.

</details>

In [ ]:
# [자가채점]
assert len(q4) == 30, f"30행이어야 합니다 (지금 {len(q4)}행)"
assert list(q4.columns) == ["orderid", "name", "bookname", "price", "saleprice", "discount"], \
    f"열 이름·순서를 확인하세요: {list(q4.columns)}"
assert (q4["discount"] == q4["price"] - q4["saleprice"]).all(), "discount 계산을 확인하세요"
keys = list(zip((-q4["discount"]).tolist(), q4["orderid"].tolist()))
assert keys == sorted(keys), \
    "할인액 내림차순, 같으면 주문번호 오름차순이어야 합니다 — 두 번째 정렬 기준을 확인하세요"
print("✅ 통과!")

## 5. 아직 한 번도 안 산 고객 (LEFT JOIN + IS NULL)

**배경**: 가입만 하고 한 번도 주문하지 않은 고객에게 첫 구매 쿠폰을 보냅니다.

**요구사항**: `customer` 를 왼쪽에 두고 `orders` 를 이어, **주문이 한 건도 없는 고객**의 `custid` · `name` 을 조회해 `q5` 에 담으세요. `custid` 오름차순으로 정렬합니다.

**예시**: 결과는 2행이고, `custid` 오름차순이므로 첫 행은 **한지우** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 왼쪽 표를 다 남기는 조인을 쓰면 짝 없는 행의 오른쪽이 빈 값이 된다
- 그 빈 값을 조건으로 삼으면 '짝이 없는 행'만 골라낼 수 있다

세부구현:
1. customer 를 FROM 에 두고 orders 를 LEFT JOIN 한다
2. WHERE 로 주문 쪽 열이 빈 값인 행만 남긴다
```

</details>

In [ ]:
# 왼쪽을 다 남기고 짝이 없으면 오른쪽이 NULL — 그 NULL 이 곧 '짝 없는 행'이라는 뜻이다
q5 = run_query("""
SELECT c.custid, c.name
FROM customer c
LEFT JOIN orders o ON o.custid = c.custid
WHERE o.orderid IS NULL
ORDER BY c.custid
""")
display(q5)

<details><summary>해설</summary>

- `LEFT JOIN` 은 왼쪽 표의 행을 하나도 버리지 않고, 짝이 없으면 오른쪽을 `NULL` 로 채웁니다.
- 그래서 `WHERE 오른쪽열 IS NULL` 이 곧 **"짝이 없는 행"** 을 뜻하는 관용구가 됩니다.
- 서브쿼리로 `WHERE custid NOT IN (SELECT custid FROM orders)` 라고 써도 같은 결과가 나옵니다.

</details>

In [ ]:
# [자가채점]
assert len(q5) == 2, f"2행이어야 합니다 (지금 {len(q5)}행)"
assert list(q5.columns) == ["custid", "name"], f"열은 custid·name 입니다: {list(q5.columns)}"
assert q5["name"].tolist() == ['한지우', '오서윤'], f"이름과 순서를 확인하세요: {q5['name'].tolist()}"
assert q5["custid"].tolist() == sorted(q5["custid"].tolist()), "custid 오름차순으로 정렬하세요"
print("✅ 통과!")

## 6. 한 명도 빠지지 않는 고객별 매출 (LEFT JOIN + 집계)

**배경**: 전체 고객 명단에 매출을 붙인 표가 필요합니다. 주문이 없는 고객도 **0원으로** 나와야 합니다.

**요구사항**: `customer` 를 왼쪽에 두고 `orders` 를 이어, `name` · 주문 건수(별칭 **`order_count`**) · 매출(별칭 **`total`**)을 조회해 `q6` 에 담으세요.

- 주문이 없는 고객은 `order_count` 가 `0`, `total` 이 `0` 이어야 합니다.
- 매출이 큰 순, 같으면 이름 오름차순으로 정렬합니다.

**예시**: 결과는 10행이고, 첫 행은 **박지성**(79000원) 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 짝이 없는 자리는 빈 값이므로, 합계를 그대로 내면 빈 값이 된다
- 빈 값을 다른 값으로 바꿔 주는 함수를 쓴다
- 건수를 셀 때도 무엇을 세느냐에 따라 0 이 되기도 하고 1 이 되기도 한다

세부구현:
1. customer 를 왼쪽에 두고 LEFT JOIN 한다
2. 건수는 주문 쪽 열을 세어 짝 없는 행이 0 이 되게 한다
3. 합계는 빈 값 대체 함수로 감싼다
4. 고객별로 묶고 정렬한다
```

</details>

In [ ]:
# 건수는 주문 쪽 열을 세야 0 이 나온다(count(*) 는 1이 된다). 합계의 NULL 은 coalesce 로 0 을 채운다
q6 = run_query("""
SELECT c.name,
       count(o.orderid)               AS order_count,
       coalesce(sum(o.saleprice), 0)  AS total
FROM customer c
LEFT JOIN orders o ON o.custid = c.custid
GROUP BY c.custid, c.name
ORDER BY total DESC, c.name
""")
display(q6)

<details><summary>해설</summary>

- `count(*)` 를 쓰면 짝이 없는 행도 **한 행으로 세어 1** 이 됩니다. `count(o.orderid)` 처럼 **오른쪽 표의 열**을 세야 0 이 나옵니다.
- `sum` 은 더할 값이 하나도 없으면 `NULL` 을 돌려줍니다. `coalesce(sum(...), 0)` 으로 0 을 채웁니다.
- `GROUP BY c.custid, c.name` 처럼 기본키를 함께 묶으면 동명이인이 있어도 안전합니다.

</details>

In [ ]:
# [자가채점]
assert len(q6) == 10, f"10행이어야 합니다 (지금 {len(q6)}행)"
assert list(q6.columns) == ["name", "order_count", "total"], \
    f"열은 name·order_count·total 입니다: {list(q6.columns)}"
assert q6["name"][0] == "박지성", "매출 1위 고객이 첫 행이어야 합니다"
assert int(q6["total"][0]) == 79000, "1위 고객의 매출을 확인하세요"
zero = q6[q6["name"].isin(['한지우', '오서윤'])]
assert (zero["order_count"] == 0).all(), "주문 없는 고객의 건수는 0 이어야 합니다 (count(*) 대신 주문 쪽 열을 세세요)"
assert (zero["total"] == 0).all(), "주문 없는 고객의 매출은 0 이어야 합니다 (coalesce 를 쓰세요)"
print("✅ 통과!")

## 7. 평균보다 비싼 주문 (스칼라 서브쿼리)

**배경**: 평균 판매가를 넘는 주문이 얼마나 되는지 봅니다.

**요구사항**: `orders` 에서 **전체 평균 판매가보다 비싼** 주문의 `orderid` · `saleprice` 를 판매가 내림차순으로 조회해 `q7` 에 담으세요. 평균값을 직접 계산해 숫자로 적지 말고 **서브쿼리로** 구하세요 (자가채점은 결과만 보지만, 숫자를 박아 두면 데이터가 바뀌는 순간 틀린 질의가 됩니다).

**예시**: 결과는 11행입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 값 하나를 돌려주는 질의는 비교연산자 오른쪽에 그대로 넣을 수 있다

세부구현:
1. 안쪽 질의를 따로 실행해 평균값을 눈으로 확인한다
2. 그 질의를 괄호로 감싸 WHERE 조건에 끼워 넣는다
3. 판매가 내림차순으로 정렬한다
```

</details>

In [ ]:
# 값 하나를 돌려주는 질의는 비교연산자 오른쪽에 그대로 넣는다 — 평균을 숫자로 박아 두지 않는다
q7 = run_query("""
SELECT orderid, saleprice
FROM orders
WHERE saleprice > (SELECT avg(saleprice) FROM orders)
ORDER BY saleprice DESC
""")
display(q7)

<details><summary>해설</summary>

- 평균을 숫자로 적어 두면 데이터가 바뀔 때마다 질의를 고쳐야 합니다. 서브쿼리로 두면 **언제 실행해도 그 시점의 평균**을 씁니다.
- 서브쿼리는 안쪽부터 읽습니다. 헷갈리면 안쪽만 따로 실행해 값을 확인하세요.

</details>

In [ ]:
# [자가채점]
assert len(q7) == 11, f"11행이어야 합니다 (지금 {len(q7)}행)"
assert list(q7.columns) == ["orderid", "saleprice"], \
    f"열은 orderid·saleprice 입니다: {list(q7.columns)}"
avg = run_query("SELECT avg(saleprice) AS a FROM orders")["a"][0]
assert (q7["saleprice"] > avg).all(), "평균 이하인 주문이 섞여 있습니다"
assert q7["saleprice"].tolist() == sorted(q7["saleprice"].tolist(), reverse=True), \
    "판매가 내림차순으로 정렬하세요"
print("✅ 통과!")

## 8. 특정 출판사 책을 산 고객 (IN 서브쿼리)

**배경**: 굿스포츠 출판사와 공동 이벤트를 합니다. 그 출판사 책을 산 적 있는 고객을 추립니다.

**요구사항**: **출판사가 `'굿스포츠'` 인 책을 한 번이라도 주문한 고객**의 `custid` · `name` 을 `custid` 오름차순으로 조회해 `q8` 에 담으세요. 고객은 **중복 없이** 한 번씩만 나와야 합니다.

**예시**: 결과는 7행입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 목록을 돌려주는 질의는 in 의 오른쪽에 넣는다
- 안쪽 질의에서 '굿스포츠 책을 산 고객 번호'를 만든다

세부구현:
1. 안쪽: 주문과 도서를 이어 출판사 조건을 걸고 고객 번호만 뽑는다
2. 바깥: 고객 표에서 그 번호에 해당하는 고객을 고른다
3. 한 고객이 여러 권 샀어도 바깥 질의는 고객 표를 보므로 한 번만 나온다
```

</details>

In [ ]:
# 안쪽 질의가 같은 고객을 여러 번 돌려줘도, 바깥은 고객 표의 행을 고르므로 중복이 없다
q8 = run_query("""
SELECT custid, name
FROM customer
WHERE custid IN (
    SELECT o.custid
    FROM orders o
    JOIN book b ON b.bookid = o.bookid
    WHERE b.publisher = '굿스포츠'
)
ORDER BY custid
""")
display(q8)

<details><summary>해설</summary>

- 안쪽 질의가 같은 고객 번호를 여러 번 돌려줘도, 바깥은 **고객 표의 행**을 고르는 것이라 고객은 한 번씩만 나옵니다. `DISTINCT` 를 안쪽에 붙여도 결과는 같습니다.
- `JOIN` 으로 풀 수도 있지만, 그때는 고객이 산 권수만큼 중복되므로 `DISTINCT` 가 필요합니다.
- 반대(그 출판사 책을 한 번도 안 산 고객)는 `NOT IN` 입니다.

</details>

In [ ]:
# [자가채점]
assert len(q8) == 7, f"7행이어야 합니다 (지금 {len(q8)}행)"
assert list(q8.columns) == ["custid", "name"], f"열은 custid·name 입니다: {list(q8.columns)}"
assert q8["custid"].is_unique, "같은 고객이 여러 번 나옵니다"
assert q8["custid"].tolist() == [1, 2, 3, 4, 5, 7, 8], \
    f"굿스포츠 책을 산 고객의 번호를 확인하세요: {q8['custid'].tolist()}"
print("✅ 통과!")

## 9. 많이 팔린 책 추리기 (JOIN + GROUP BY)

**배경**: 매대에 올릴 후보를 뽑습니다. 이 서점은 상위권이 **동점**이라, 순위를 말하려면 동점을 어떻게 다룰지부터 정해야 합니다.

**요구사항**: 주문과 도서를 이어 **책별 판매 부수**를 세고, 많이 팔린 순, 같으면 **책 제목 오름차순**으로 정렬해 **위에서 3권만** `q9` 에 담으세요. 열은 `bookname` 과 부수(별칭 **`sold`**) 입니다.

**예시**: 결과는 3행이고 첫 행은 **Olympic Champions**(4부) 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 조인해서 붙인 결과를 다시 묶어 센다

세부구현:
1. 주문과 도서를 잇는다
2. 책별로 묶고 센다
3. 두 기준으로 정렬한 뒤 위에서 세 개만 자른다
```

</details>

In [ ]:
# 기본키를 함께 묶으면 제목이 같은 다른 책이 있어도 섞이지 않는다
q9 = run_query("""
SELECT b.bookname,
       count(*) AS sold
FROM orders o
JOIN book b ON b.bookid = o.bookid
GROUP BY b.bookid, b.bookname
ORDER BY sold DESC, b.bookname
LIMIT 3
""")
display(q9)

<details><summary>해설</summary>

- `GROUP BY b.bookid, b.bookname` 처럼 **기본키를 함께** 묶으면 같은 제목의 다른 책이 있어도 안 섞입니다.
- 동점이 있을 때 순서가 흔들리지 않도록 **두 번째 정렬 기준**을 두는 것이 좋은 습관입니다.

</details>

In [ ]:
# [자가채점]
assert len(q9) == 3, f"3행이어야 합니다 (지금 {len(q9)}행)"
assert list(q9.columns) == ["bookname", "sold"], f"열은 bookname·sold 입니다: {list(q9.columns)}"
assert q9["bookname"][0] == "Olympic Champions", \
    f"첫 행은 Olympic Champions 여야 합니다 (지금 {q9['bookname'][0]})"
assert int(q9["sold"][0]) == 4, "판매 부수를 확인하세요"
print("✅ 통과!")

## 10. 7월 매출만 따로 (기간 조건 + 집계)

**배경**: 7월 실적만 떼어 고객별로 봅니다.

**요구사항**: `orders` 에서 **주문일이 2026년 7월**인 주문만 골라 고객별로 묶고, `custid` · 건수(별칭 **`order_count`**) · 매출(별칭 **`total`**)을 조회해 `q10` 에 담으세요. 매출이 큰 순, 같으면 `custid` 오름차순으로 정렬합니다.

**예시**: 7월 주문은 모두 17건이고, 결과는 7행입니다.

> `orderdate` 는 `'2026-07-01'` 같은 글자로 저장돼 있습니다. 글자도 `BETWEEN` 으로 범위 비교가 됩니다.

<details><summary>힌트</summary>

```text
접근방법:
- 묶기 전에 행을 거르는 조건이므로 WHERE 를 쓴다

세부구현:
1. WHERE 로 날짜 범위를 건다 (양끝 포함)
2. 고객별로 묶어 건수와 합계를 낸다
3. 두 기준으로 정렬한다
```

</details>

In [ ]:
# 'YYYY-MM-DD' 글자는 글자 순서가 곧 날짜 순서라 BETWEEN 이 통한다. WHERE 는 묶기 전에 걸린다
q10 = run_query("""
SELECT custid,
       count(*)        AS order_count,
       sum(saleprice)  AS total
FROM orders
WHERE orderdate BETWEEN '2026-07-01' AND '2026-07-31'
GROUP BY custid
ORDER BY total DESC, custid
""")
display(q10)

<details><summary>해설</summary>

- 날짜를 `'YYYY-MM-DD'` 형식의 글자로 저장하면 **글자 순서가 곧 날짜 순서**라 `BETWEEN` · `ORDER BY` 가 그대로 통합니다. 형식을 섞어 쓰면 이 성질이 깨집니다.
- `WHERE` 는 묶기 전에 걸립니다. 여기서 거른 행만 집계에 들어갑니다.

</details>

In [ ]:
# [자가채점]
assert len(q10) == 7, f"7행이어야 합니다 (지금 {len(q10)}행)"
assert list(q10.columns) == ["custid", "order_count", "total"], \
    f"열은 custid·order_count·total 입니다: {list(q10.columns)}"
assert int(q10["order_count"].sum()) == 17, \
    f"건수의 합이 7월 주문 17건과 같아야 합니다 (지금 {int(q10['order_count'].sum())})"
assert q10["total"].tolist() == sorted(q10["total"].tolist(), reverse=True), \
    "매출이 큰 순으로 정렬하세요"
assert int(q10["custid"][0]) == 3 and int(q10["order_count"][0]) == 4, \
    f"1위는 custid 3(4건)입니다: {q10.iloc[0].tolist()}"
assert int(q10["total"][0]) == 53500, \
    f"1위 고객의 7월 매출은 53500 입니다: {q10['total'][0]}"
print("✅ 통과!")

## 11. 아직 덜 사 간 고객 (LEFT JOIN + GROUP BY + HAVING)

**배경**: 재구매 안내를 보낼 대상은 **주문이 2건 이하인 고객**입니다. 여기에는 **한 번도 사지 않은 고객도 포함**되어야 합니다 — 그 사람들이야말로 안내가 가장 필요합니다.

**요구사항**: `customer` 를 왼쪽에 두고 `orders` 를 이어, 고객별 주문 건수를 세고 **2건 이하인 고객만** 남겨 `q11` 에 담으세요. 열은 `name` 과 주문 건수(별칭 **`order_count`**) 이고, 건수가 많은 순, 같으면 **이름 오름차순**으로 정렬합니다.

**예시**: 결과는 3행이고 첫 행은 **김서준**(2건) 입니다.

> 5번·6번에서 쓴 두 가지를 한 문장에 겹쳐 쓰는 문제입니다 — **짝이 없어도 남기는 조인**과 **묶은 뒤에 거는 조건**.

<details><summary>힌트</summary>

```text
접근방법:
- 한 건도 주문하지 않은 고객을 살리려면 왼쪽을 다 남기는 조인이어야 한다
- 0건이 0 으로 세어지려면 무엇을 세느냐가 중요하다
- 묶은 결과에 거는 조건은 WHERE 가 아니다

세부구현:
1. customer 를 FROM 에 두고 orders 를 이어 붙인다
2. 고객별로 묶고 주문 쪽 열을 세어 건수를 낸다
3. 그 건수에 조건을 건다
4. 두 기준으로 정렬한다
```

</details>

In [ ]:
# 주문 0건 고객을 살리려면 LEFT JOIN, 0 으로 세려면 주문 쪽 열, 묶은 뒤 조건은 HAVING
q11 = run_query("""
SELECT c.name,
       count(o.orderid) AS order_count
FROM customer c
LEFT JOIN orders o ON o.custid = c.custid
GROUP BY c.custid, c.name
HAVING count(o.orderid) <= 2
ORDER BY order_count DESC, c.name
""")
display(q11)

<details><summary>해설</summary>

- `JOIN`(안쪽 조인)으로 쓰면 **주문이 0건인 고객이 통째로 사라져** 정작 가장 필요한 대상이 빠집니다. 짝이 없어도 왼쪽을 남기려면 `LEFT JOIN` 입니다.
- `count(*)` 는 짝 없는 행도 한 행으로 세어 **1** 이 됩니다. `count(o.orderid)` 처럼 오른쪽 표의 열을 세야 0 이 나옵니다(6번과 같은 함정).
- `HAVING` 은 묶은 **뒤**에 겁니다. `WHERE count(...) <= 2` 는 오류입니다.

</details>

In [ ]:
# [자가채점]
assert len(q11) == 3, f"3행이어야 합니다 (지금 {len(q11)}행)"
assert list(q11.columns) == ["name", "order_count"], \
    f"열은 name·order_count 입니다: {list(q11.columns)}"
assert q11["name"].tolist() == ['김서준', '오서윤', '한지우'], \
    f"이름과 순서를 확인하세요: {q11['name'].tolist()}"
assert q11["order_count"].tolist() == [2, 0, 0], \
    f"건수를 확인하세요: {q11['order_count'].tolist()}"
zero = q11[q11["order_count"] == 0]["name"].tolist()
assert sorted(zero) == ['오서윤', '한지우'], (
    "주문이 0건인 고객이 빠졌습니다 — 안쪽 조인이 아니라 LEFT JOIN 이어야 하고, "
    f"건수는 주문 쪽 열을 세야 0 이 됩니다: {zero}")
print("✅ 통과!")

## 12. 7월 굿스포츠 매출 (3표 JOIN + 기간 조건 + 집계)

**배경**: 굿스포츠 출판사와의 7월 공동 이벤트 정산을 합니다. **7월에 굿스포츠 책을 산 고객이 각각 얼마를 썼는지** 뽑아야 합니다.

**요구사항**: `orders` · `customer` · `book` 세 표를 이어, **출판사가 `'굿스포츠'` 이고 주문일이 2026년 7월**인 주문만 골라 고객별 매출을 구해 `q12` 에 담으세요. 열은 `name` 과 매출(별칭 **`total`**) 이고, 매출이 큰 순, 같으면 **이름 오름차순**으로 정렬합니다.

**예시**: 결과는 4행이고 1위는 **김연아**(8000원) 입니다. 매출이 같은 고객이 있으니 두 번째 정렬 기준을 꼭 넣으세요.

> `orderdate` 는 `'2026-07-01'` 같은 글자입니다 — `BETWEEN` 으로 범위를 잡을 수 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 4번에서 한 3표 잇기에, 10번에서 한 기간 조건과 집계를 겹친다
- 조건 두 개를 모두 만족해야 하므로 AND 로 잇는다
- 거르는 일이 먼저, 묶는 일이 나중이다

세부구현:
1. 주문을 가운데 두고 고객·도서를 각각 잇는다
2. WHERE 에 출판사 조건과 날짜 범위 조건을 함께 건다
3. 고객별로 묶어 판매가를 합한다
4. 두 기준으로 정렬한다
```

</details>

In [ ]:
# 출판사 조건은 book 에, 날짜 조건은 orders 에 있다 — 이어 붙였기에 한 WHERE 에 나란히 쓴다
q12 = run_query("""
SELECT c.name,
       sum(o.saleprice) AS total
FROM orders o
JOIN customer c ON c.custid = o.custid
JOIN book b ON b.bookid = o.bookid
WHERE b.publisher = '굿스포츠'
  AND o.orderdate BETWEEN '2026-07-01' AND '2026-07-31'
GROUP BY c.custid, c.name
ORDER BY total DESC, c.name
""")
display(q12)

<details><summary>해설</summary>

- 조건은 **묶기 전**에 걸립니다. 여기서 걸러 남은 주문만 합계에 들어갑니다.
- 출판사 조건은 `book` 에, 날짜 조건은 `orders` 에 있습니다 — 조인해 한 줄로 모았기 때문에 두 표의 조건을 한 `WHERE` 에 나란히 쓸 수 있습니다.
- `HAVING` 으로 옮겨도 sqlite 에서는 답이 나오지만, **묶기 전에 거를 수 있는 조건은 `WHERE` 에** 두는 것이 옳고 빠릅니다.

</details>

In [ ]:
# [자가채점]
assert len(q12) == 4, f"4행이어야 합니다 (지금 {len(q12)}행)"
assert list(q12.columns) == ["name", "total"], f"열은 name·total 입니다: {list(q12.columns)}"
assert q12["name"].tolist() == ['김연아', '박세리', '박지성', '장미란'], \
    f"이름과 순서를 확인하세요 — 동점은 이름 오름차순입니다: {q12['name'].tolist()}"
assert [int(v) for v in q12["total"]] == [8000, 6500, 6000, 6000], \
    f"매출을 확인하세요: {q12['total'].tolist()}"
print("✅ 통과!")

## 13. 우수고객 명단 만들기 (CREATE + INSERT INTO SELECT)

**배경**: 매출 상위 고객을 따로 관리할 표가 필요합니다. 조회 결과를 **다른 표에 그대로 복사**하는 방법을 씁니다.

**요구사항**

1. `vip_customer` 표를 `STRICT` 로 만드세요. 열은 `custid`(integer, 기본키) · `name`(text, 비어 있으면 안 됨) · `total`(int, 비어 있으면 안 됨) 입니다.
2. **누적 매출이 50000 이상인 고객**을 조회해 그 결과를 `vip_customer` 에 넣으세요. `INSERT INTO 표 (열들) SELECT ...` 형태를 씁니다.
3. `vip_customer` 를 매출 내림차순으로 조회해 확인하세요.

**예시**: 4명이 들어가고, 매출 1위는 **박지성** 입니다.

**예시**

```
INSERT INTO 표 (열1, 열2)
SELECT ... FROM ... WHERE ...;
```

<details><summary>힌트</summary>

```text
접근방법:
- 조회 결과를 그대로 다른 표에 넣을 수 있다. VALUES 자리에 SELECT 를 쓴다
- 넣을 열 개수·순서와 SELECT 가 돌려주는 열 개수·순서가 1:1 로 맞아야 한다

세부구현:
1. 표를 먼저 만든다
2. 고객과 주문을 이어 고객별 매출을 구한다
3. HAVING 으로 매출 조건을 건다
4. 그 SELECT 를 INSERT INTO ... SELECT 형태로 감싼다
5. 넣은 결과를 조회해 확인한다
```

</details>

In [ ]:
# 1) 담을 표를 먼저 만든다 — 넣을 값에 맞춰 타입과 규칙을 정한다
run_sql("""
CREATE TABLE vip_customer (
    custid integer PRIMARY KEY,
    name   text NOT NULL,
    total  int  NOT NULL
) STRICT
""")

# 2) VALUES 자리에 SELECT 를 쓴다 — 조회 결과가 통째로 들어간다.
#    넣을 열 순서와 SELECT 의 열 순서가 1:1 로 맞아야 한다(어긋나도 오류가 안 난다)
run_sql("""
INSERT INTO vip_customer (custid, name, total)
SELECT c.custid, c.name, sum(o.saleprice)
FROM customer c
JOIN orders o ON o.custid = c.custid
GROUP BY c.custid, c.name
HAVING sum(o.saleprice) >= 50000
""")

# 3) 넣은 결과를 조회해 확인한다 — 조용히 잘못 들어가는 실수를 여기서 잡는다
display(run_query("SELECT * FROM vip_customer ORDER BY total DESC"))

<details><summary>해설</summary>

- `INSERT INTO ... SELECT` 는 **조회 결과를 통째로** 넣습니다. 한 행씩 반복문을 돌 필요가 없습니다.
- 넣는 열 목록과 `SELECT` 의 열 순서가 어긋나면 엉뚱한 값이 들어갑니다 — 오류가 나지 않고 **조용히** 잘못되기 쉬우니 조회로 꼭 확인하세요.
- `HAVING` 대신 `WHERE sum(...) >= 50000` 은 오류입니다(집계는 묶은 뒤에 판정).

</details>

In [ ]:
# [자가채점]
cols = run_query("pragma table_info(vip_customer)")
assert list(cols["name"]) == ["custid", "name", "total"], \
    f"열 이름·순서를 확인하세요: {list(cols['name'])}"
ddl = run_query("SELECT sql FROM sqlite_master WHERE name = 'vip_customer'")["sql"][0].lower()
assert "strict" in ddl, "표 끝에 STRICT 를 붙이세요"
assert int(cols[cols["name"] == "custid"]["pk"].iloc[0]) == 1, "custid 를 기본키로 두세요"
for col in ["name", "total"]:
    assert int(cols[cols["name"] == col]["notnull"].iloc[0]) == 1, \
        f"{col} 에 NOT NULL 을 거세요"
rows = run_query("SELECT name, total FROM vip_customer ORDER BY total DESC")
assert len(rows) == 4, f"4행이어야 합니다 (지금 {len(rows)}행)"
assert rows["name"].tolist() == ['박지성', '장미란', '이하은', '추신수'], \
    f"이름과 순서를 확인하세요: {rows['name'].tolist()}"
assert (rows["total"] >= 50000).all(), "매출 5만 미만인 고객이 섞여 있습니다"
print("✅ 통과!")

## 14. 명단을 7월 기준으로 다시 계산 (UPDATE + 상관 서브쿼리)

> 이 문제는 **13번에서 만든 `vip_customer`** 를 씁니다. 13번을 먼저 푸세요.

**배경**: 명단의 `total` 은 **전체 기간** 누적 매출입니다. 7월 실적으로 시상하기로 해서, 각 고객의 `total` 을 **7월 매출만**으로 다시 채워야 합니다.

**요구사항**: `vip_customer` 의 모든 행을 갱신하세요. 각 행의 `total` 은 **그 고객이 2026년 7월에 쓴 금액**이 됩니다. **7월에 한 번도 사지 않은 고객은 `0`** 이어야 합니다.

- 명단에서 **행을 지우지 마세요.** 이 문제는 값만 바꿉니다(빼는 일은 15번에서 합니다).
- 갱신한 뒤 매출 내림차순으로 조회해 확인하세요.

**예시**: 4명 그대로이고, 1위가 **장미란**(53500원)로 바뀝니다 — 전체 기간 1위였던 박지성 은 순위가 내려갑니다.

<details><summary>힌트</summary>

```text
접근방법:
- 한 행씩 값을 계산해야 한다. 바깥 표의 그 행을 가리키며 도는 서브쿼리를 쓴다
- 서브쿼리 안에서 바깥 표의 열을 그대로 부를 수 있다 (표 이름을 적어 준다)
- 7월에 산 것이 하나도 없으면 합계가 빈 값이 된다 — 빈 값을 0 으로 바꿔 주는 함수가 필요하다

세부구현:
1. UPDATE 표 SET 열 = (서브쿼리) 모양으로 적는다
2. 서브쿼리에서 주문 표를 그 고객으로 좁히고 기간 조건을 건다
3. 합계를 빈 값 대체 함수로 감싼다
4. 갱신 뒤 조회해 순위가 바뀐 것을 확인한다
```

</details>

In [ ]:
# 바깥 표의 그 행(vip_customer.custid)을 가리키며 도는 상관 서브쿼리다.
# 7월 주문이 하나도 없으면 sum 이 NULL 이라, coalesce 로 0 을 채워야 NOT NULL 에 걸리지 않는다.
run_sql("""
UPDATE vip_customer
SET total = (
    SELECT coalesce(sum(o.saleprice), 0)
    FROM orders o
    WHERE o.custid = vip_customer.custid
      AND o.orderdate BETWEEN '2026-07-01' AND '2026-07-31'
)
""")

display(run_query("SELECT * FROM vip_customer ORDER BY total DESC"))

<details><summary>해설</summary>

- **상관 서브쿼리**입니다. 안쪽 질의가 `vip_customer.custid` 를 참조하므로 **바깥 행마다 한 번씩** 다시 계산됩니다. 7번의 스칼라 서브쿼리(전체 평균)는 한 번만 계산되던 것과 다릅니다.
- `coalesce` 를 빼면 7월 주문이 없는 고객의 `total` 이 `NULL` 이 되고, 13번에서 **자기가 건 `NOT NULL` 제약**에 막혀 `IntegrityError` 가 납니다. 제약이 잘못된 갱신을 실행 시점에 잡아 주는 장면입니다.
- `WHERE` 를 빠뜨리면 **모든 행**이 갱신됩니다. 여기서는 그것이 의도(전원 갱신)지만, 일부만 바꿀 때는 `UPDATE ... WHERE` 를 반드시 붙이세요.
- 실무에서는 이렇게 **집계 결과를 표에 적어 두는 것**을 비정규화라고 부릅니다. 빠르지만 원본이 바뀌면 다시 계산해 줘야 합니다 — 지금 한 일이 바로 그 재계산입니다.

</details>

In [ ]:
# [자가채점]
assert run_query("SELECT count(*) AS n FROM sqlite_master "
                 "WHERE name = 'vip_customer'")["n"][0] == 1, \
    "vip_customer 표가 없습니다 — 13번을 먼저 푸세요"
rows = run_query("SELECT name, total FROM vip_customer ORDER BY total DESC, name")
assert len(rows) == 4, \
    f"4행이어야 합니다 — 이 문제는 값만 바꿉니다 (지금 {len(rows)}행)"
assert rows["name"].tolist() == ['장미란', '추신수', '박지성', '이하은'], \
    f"이름과 순서를 확인하세요: {rows['name'].tolist()}"
assert [int(v) for v in rows["total"]] == [53500, 44500, 39000, 0], \
    f"7월 매출로 갱신됐는지 확인하세요: {rows['total'].tolist()}"
assert int(rows[rows["name"] == "이하은"]["total"].iloc[0]) == 0, \
    "7월에 산 적 없는 고객(이하은)의 total 은 0 이어야 합니다"
print("✅ 통과!")

## 15. 7월에 안 산 고객은 명단에서 빼기 (DELETE + 서브쿼리)

> 이 문제는 **14번까지 끝낸 상태**에서 이어집니다.

**배경**: 7월 시상 명단이므로 **7월에 한 건도 사지 않은 고객**은 명단에 남을 이유가 없습니다.

**요구사항**: `vip_customer` 에서 **2026년 7월에 주문이 하나도 없는 고객**의 행을 지우세요. 지운 뒤 남은 명단을 매출 내림차순으로 조회해 확인합니다.

**예시**: 4명에서 **3명**으로 줄고, **이하은** 이 빠집니다.

> `total` 이 0 인 행을 지우는 것도 결과는 같습니다. 하지만 그것은 **14번을 푼 뒤에만** 통하는 우연입니다. **7월 주문이 없다**는 조건 자체를 `orders` 에서 확인해 지우세요.

<details><summary>힌트</summary>

```text
접근방법:
- 지울 대상을 목록으로 만들어 '그 목록에 없는' 행을 고른다
- 8번에서 쓴 목록 서브쿼리의 반대 형태다

세부구현:
1. 안쪽: 7월에 주문한 고객 번호 목록을 만든다
2. 바깥: 그 목록에 없는 고객의 행을 지운다
3. 지우기 전에 같은 조건으로 SELECT 해 대상을 확인한다
4. 남은 명단을 조회한다
```

</details>

In [ ]:
# 1) 지워질 대상을 먼저 눈으로 확인한다
display(run_query("""
SELECT * FROM vip_customer
WHERE custid NOT IN (
    SELECT custid FROM orders
    WHERE orderdate BETWEEN '2026-07-01' AND '2026-07-31'
)
"""))

# 2) 같은 조건으로 지운다 — '7월 주문이 없다'를 orders 에서 직접 확인한다
run_sql("""
DELETE FROM vip_customer
WHERE custid NOT IN (
    SELECT custid FROM orders
    WHERE orderdate BETWEEN '2026-07-01' AND '2026-07-31'
)
""")

# 3) 남은 명단 확인
display(run_query("SELECT * FROM vip_customer ORDER BY total DESC"))

<details><summary>해설</summary>

- `NOT IN (서브쿼리)` 는 8번의 `IN` 을 뒤집은 것입니다. '7월에 주문한 고객'을 만든 뒤 **거기에 없는** 사람을 고릅니다.
- `WHERE total = 0` 으로도 같은 행이 지워지지만, 그건 14번을 푼 **뒤에만** 맞는 조건입니다. 조건은 **지우려는 이유 그대로** 쓰는 것이 안전합니다 — 나중에 값이 바뀌어도 의미가 유지됩니다.
- `NOT IN` 은 목록에 `NULL` 이 섞이면 **아무것도 못 지웁니다**(비교 결과가 참이 되지 않습니다). 여기서는 `orders.custid` 에 `NULL` 이 없어 안전합니다.
- 지우기 전에 같은 조건으로 `SELECT` 를 돌려 보는 습관은 LV1 에서 익힌 그대로입니다.

</details>

In [ ]:
# [자가채점]
assert run_query("SELECT count(*) AS n FROM sqlite_master "
                 "WHERE name = 'vip_customer'")["n"][0] == 1, \
    "vip_customer 표가 없습니다 — 13번을 먼저 푸세요"
rows = run_query("SELECT name, total FROM vip_customer ORDER BY total DESC, name")
assert len(rows) == 3, f"3행이 남아야 합니다 (지금 {len(rows)}행)"
assert rows["name"].tolist() == ['장미란', '추신수', '박지성'], \
    f"남은 이름과 순서를 확인하세요: {rows['name'].tolist()}"
assert [int(v) for v in rows["total"]] == [53500, 44500, 39000], \
    "남은 행의 total 까지 바뀌었습니다 — 이 문제는 지우기만 합니다"
assert run_query("SELECT count(*) AS n FROM customer")["n"][0] == 10, \
    "customer 표의 행이 사라졌습니다 — 지울 대상은 vip_customer 뿐입니다"
assert run_query("SELECT count(*) AS n FROM orders")["n"][0] == 30, \
    "orders 표의 행이 사라졌습니다 — 지울 대상은 vip_customer 뿐입니다"
print("✅ 통과!")

## 16. 실습 정리하기 (DROP TABLE)

**배경**: 시상 명단을 다 뽑았습니다. `vip_customer` 는 **집계용으로 잠깐 만든 표**라 역할이 끝나면 남겨 둘 이유가 없습니다.

**요구사항**: `vip_customer` 표를 지우세요. 표 자체가 없어져야 합니다.

- `customer` · `book` · `orders` 는 그대로 두세요.
- 이 표는 13번에서 여러분이 `CREATE TABLE` 로 만든 것이라, **지우고 나면 13번부터 다시 풀어 볼 수 있습니다.** (지우지 않고 13번을 다시 실행하면 `table vip_customer already exists` 로 막힙니다.)

> **`DELETE` 와 무엇이 다른가**: `DELETE FROM vip_customer` 는 **행만** 지우고 빈 표는 그대로 남습니다. 지금 필요한 것은 **표 자체를 없애는** 명령입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 행을 지우는 명령과 표를 없애는 명령은 다르다

세부구현:
1. 표를 없애는 명령을 실행한다
2. sqlite_master 를 조회해 그 이름이 사라졌는지 확인한다
```

</details>

In [ ]:
# DELETE 는 행만 지우고 표는 남는다. 표 자체를 없애는 것은 DROP TABLE 이다.
run_sql("DROP TABLE vip_customer")

# 남아 있는 표 목록으로 확인한다 (실습 기준 표 셋만 남아야 한다)
display(run_query("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"))

<details><summary>해설</summary>

- `DROP TABLE` 은 **구조까지** 없앱니다. 되돌리려면 `CREATE TABLE` 부터 다시 해야 합니다.
- 그래서 실무에서는 지우기 전에 **정말 임시 표인지** 확인합니다. 이름에 `tmp_`·`stg_` 같은 접두어를 붙여 두는 관례가 그래서 생겼습니다.
- `DROP TABLE IF EXISTS vip_customer` 라고 쓰면 표가 없어도 오류가 나지 않습니다 — 리셋 스크립트가 늘 이 형태를 쓰는 이유입니다.

</details>

In [ ]:
# [자가채점]
tables = run_query("SELECT name FROM sqlite_master WHERE type = 'table'")["name"].tolist()
assert "vip_customer" not in tables, \
    "vip_customer 표가 아직 있습니다 — 행만 지운 것은 아닌지 확인하세요"
for t in ["customer", "book", "orders"]:
    assert t in tables, f"{t} 표까지 지워졌습니다 — 지울 표는 vip_customer 하나입니다"
assert run_query("SELECT count(*) AS n FROM orders")["n"][0] == 30, \
    "orders 의 행이 사라졌습니다"
print("✅ 통과!")

## 다 풀었다면

- 자가채점이 모두 `✅ 통과!` 인지 확인하세요.
- 11번·12번 질의를 다시 읽어 보세요 — **관계(JOIN) · 조건(WHERE) · 묶기(GROUP BY) · 묶은 뒤의 조건(HAVING)** 이 한 문장 안에서 각자 제 자리를 지키고 있습니다. SQL 을 읽는 힘은 이 자리 감각에서 나옵니다.
- 이어서 **과제 LV3** 로 넘어갑니다 — Supabase 와 pgvector 로 **뜻으로 찾아 근거를 붙여 답하는 RAG** 를 직접 만듭니다.